In [1]:
import requests
import pandas as pd
from datetime import datetime
import time

# Links

In [2]:
links = [
    "https://www.reddit.com/r/phcareers/comments/17r76bz/ive_been_job_searching_for_more_than_a_month_now/",
    "https://www.reddit.com/r/phcareers/comments/1b7a0m2/i_never_knew_looking_for_work_would_be_this_hard/",
    "https://www.reddit.com/r/phcareers/comments/1iy2bg1/mahirap_pala_ang_job_hunting_especially_if_from/",
    "https://www.reddit.com/r/phcareers/comments/zfs15j/resigning_even_without_a_new_job/",
    "https://www.reddit.com/r/phcareers/comments/14rylp9/ayoko_ko_na_magtrabaho_sa_government_natin/",
    "https://www.reddit.com/r/phcareers/comments/1d7wjm4/my_goal_was_to_have_a_new_job_by_june_this_june_i/",
    "https://www.reddit.com/r/phcareers/comments/1i3x4m1/how_is_our_job_hunting_early_this_2025_so_far/",
    "https://www.reddit.com/r/phcareers/comments/1jsxgck/random_help_thread_april_07_to_april_13_2025/",
    "https://www.reddit.com/r/phcareers/comments/190vcqj/random_help_thread_january_08_to_january_14_2024/",
    "https://www.reddit.com/r/phcareers/comments/1j7d686/the_recruiter_who_ghosted_me_suddenly_came_back/",
    "https://www.reddit.com/r/phcareers/comments/171254h/i_was_ghosted_after_having_been_scheduled_for_a/",
    "https://www.reddit.com/r/phcareers/comments/15rd83t/ghosted_by_an_employer_who_promised_me_sht_what/",
    "https://www.reddit.com/r/phcareers/comments/1hxwi6g/thoughts_on_ghostingbeing_ghosted_during_job/",
    "https://www.reddit.com/r/phcareers/comments/16tt3y5/why_is_my_jo_taking_so_long_am_i_ghosted_by_the/",
    "https://www.reddit.com/r/phcareers/comments/16zcg4k/how_to_deal_with_ghosting_recruiters/",
    "https://www.reddit.com/r/phcareers/comments/1k7eafo/why_do_some_companies_ignore_applicants_after_the/",
    "https://www.reddit.com/r/phcareers/comments/1imwgne/i_was_ghosted_by_hr_after_they_told_me_i_got_the/",
    "https://www.reddit.com/r/phcareers/comments/1490t09/109_applications_in_and_still_no_luck_landing_a/",
    "https://www.reddit.com/r/phcareers/comments/17oxhgu/unknown_number_called_me_offering_an_online_job/",
    "https://www.reddit.com/r/phcareers/comments/114hij0/should_you_be_emailed_back_by_recruiters_to_tell/",
    "https://www.reddit.com/r/phcareers/comments/16n0k3r/sr_recruitment_manager_here_to_answer_your/",
    "https://www.reddit.com/r/phcareers/comments/12b1fja/unemployed_lost_and_losing_hope/",
    "https://www.reddit.com/r/phcareers/comments/116wamy/hundreds_of_applications_dozens_of_rejections/"
]

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

def assign_theme(text):
    text = text.lower()
    if "ghost" in text or "no reply" in text or "no response" in text:
        return "GHOSTING"
    elif "hard" in text or "difficult" in text or "draining" in text:
        return "EMOTIONAL"
    elif "scam" in text:
        return "SCAM"
    elif "resume" in text:
        return "VISIBILITY"
    elif "application" in text:
        return "PROCESS"
    else:
        return "OTHER"

rows = []

for url in links:
    json_url = url.rstrip("/") + ".json?raw_json=1"

    success = False

    for attempt in range(3):
        try:
            res = requests.get(json_url, headers=headers)

            print(res.status_code, url)

            if res.status_code == 200 and res.text.strip():
                data = res.json()
                success = True
                break
            else:
                time.sleep(3)

        except:
            time.sleep(3)

    if not success:
        print(f"Skipped: {url}")
        continue

    # -------------------------
    # POST
    # -------------------------
    try:
        post = data[0]["data"]["children"][0]["data"]
    except:
        continue

    post_content = post["selftext"].strip() if post["selftext"] else post["title"]

    rows.append({
        "Source Subreddit": post.get("subreddit", ""),
        "Post URL": url,
        "Post Title": post.get("title", ""),
        "Post or Comment": "post",
        "Content (Full Text)": post_content,
        "Date Posted": datetime.fromtimestamp(post["created_utc"]).strftime("%m/%Y"),
        "Upvotes / Score": post.get("score", 0),
        "Theme Tag": assign_theme(post_content),
        "Platform Mentioned": "None",
        "Philippine Context?": "Yes" if post.get("subreddit","").lower().startswith("ph") else "No",
        "Notable Quote": post_content[:200],
        "Intern Notes": ""
    })

    # -------------------------
    # COMMENTS
    # -------------------------
    try:
        comments = data[1]["data"]["children"]
    except:
        comments = []

    for comment in comments:
        if comment.get("kind") != "t1":
            continue

        c = comment["data"]
        body = c.get("body", "").strip()

        # skip deleted
        if not body or body in ["[deleted]", "[removed]"]:
            continue

        rows.append({
            "Source Subreddit": post.get("subreddit", ""),
            "Post URL": url,
            "Post Title": post.get("title", ""),
            "Post or Comment": "comment",
            "Content (Full Text)": body,
            "Date Posted": datetime.fromtimestamp(c["created_utc"]).strftime("%m/%Y"),
            "Upvotes / Score": c.get("score", 0),
            "Theme Tag": assign_theme(body),
            "Platform Mentioned": "None",
            "Philippine Context?": "Yes" if post.get("subreddit","").lower().startswith("ph") else "No",
            "Notable Quote": body[:200],
            "Intern Notes": ""
        })

    time.sleep(4)

df = pd.DataFrame(rows)
df.to_csv("reddit_data.csv", index=False, encoding="utf-8-sig")

print("Done. Rows collected:", len(df))

200 https://www.reddit.com/r/phcareers/comments/17r76bz/ive_been_job_searching_for_more_than_a_month_now/
200 https://www.reddit.com/r/phcareers/comments/1b7a0m2/i_never_knew_looking_for_work_would_be_this_hard/
200 https://www.reddit.com/r/phcareers/comments/1iy2bg1/mahirap_pala_ang_job_hunting_especially_if_from/
200 https://www.reddit.com/r/phcareers/comments/zfs15j/resigning_even_without_a_new_job/
200 https://www.reddit.com/r/phcareers/comments/14rylp9/ayoko_ko_na_magtrabaho_sa_government_natin/
200 https://www.reddit.com/r/phcareers/comments/1d7wjm4/my_goal_was_to_have_a_new_job_by_june_this_june_i/
200 https://www.reddit.com/r/phcareers/comments/1i3x4m1/how_is_our_job_hunting_early_this_2025_so_far/
200 https://www.reddit.com/r/phcareers/comments/1jsxgck/random_help_thread_april_07_to_april_13_2025/
200 https://www.reddit.com/r/phcareers/comments/190vcqj/random_help_thread_january_08_to_january_14_2024/
200 https://www.reddit.com/r/phcareers/comments/1j7d686/the_recruiter_who_gh

# Comments for r/jobs

In [8]:
jobs_links = [
    "https://www.reddit.com/r/jobs/comments/1m6kdzc/no_one_is_hiring_me/",
    "https://www.reddit.com/r/jobs/comments/1msay78/the_job_market_isnt_just_broken_its_breaking/",
    "https://www.reddit.com/r/jobs/comments/1ly043a/i_got_fired_they_asked_me_to_give_my_100_at_the/",
    "https://www.reddit.com/r/jobs/comments/1pddrfk/quit_job_and_was_ignored/",
    "https://www.reddit.com/r/jobs/comments/1ldmqil/psa_dont_ignore_when_someone_offers_to_introduce/",
    "https://www.reddit.com/r/jobs/comments/1m7c7nn/job_market_is_horrendous_us/",
    "https://www.reddit.com/r/jobs/comments/1m5wl2z/getting_a_job_is_so_hard_right_now/",
    "https://www.reddit.com/r/jobs/comments/1s4frb4/i_got_a_job_offer_and_im_devastated/",
    "https://www.reddit.com/r/jobs/comments/1pgw7j9/how_should_i_respond_to_this_email_despite_my/",
    "https://www.reddit.com/r/jobs/comments/1poxk1s/after_a_14_month_struggle_i_was_ready_to_give_it/",
    "https://www.reddit.com/r/jobs/comments/1nis2km/my_wife_does_not_understand_the_state_of_the_job/",
    "https://www.reddit.com/r/jobs/comments/1mlurwz/after_hundreds_of_job_applications_and_being/",
    "https://www.reddit.com/r/jobs/comments/1pvsmi1/i_wish_there_were_regulations_that_made_ghost/",
    "https://www.reddit.com/r/jobs/comments/1rb0zp2/it_finally_happened_but_dang_why_is_the_job/",
    "https://www.reddit.com/r/jobs/comments/1loyu21/wife_laid_off/",
    "https://www.reddit.com/r/jobs/comments/1occcoo/dont_leave_jobs_ensure_you_have_a_backup_plan/",
    "https://www.reddit.com/r/jobs/comments/1kr5sa9/war_is_over_i_got_the_jobbbb/",
    "https://www.reddit.com/r/jobs/comments/1pu1dfs/bringing_parents_to_job_interview/",
    "https://www.reddit.com/r/jobs/comments/1ntx0iy/finally_good_news_i_got_the_job/",
    "https://www.reddit.com/r/jobs/comments/1p4y7kt/update_2_okay_guys_so_news_came_up_yesterday_the/",
    "https://www.reddit.com/r/jobs/comments/1smrk75/where_is_america_because_i_dont_see_it_anymore/",
    "https://www.reddit.com/r/jobs/comments/1r1gyga/to_all_people_associated_with_hiring_basic_human/",
    "https://www.reddit.com/r/jobs/comments/1pf5hso/a_year_later_they_called_to_say_they_finally_got/",
    "https://www.reddit.com/r/jobs/comments/1k2uw1t/remote_job_destroyed_my_life_unexpectedly/",
    "https://www.reddit.com/r/jobs/comments/1m0pudf/careerbuilder_and_monster_are_bankrupt_somehow/",
    "https://www.reddit.com/r/jobs/comments/1rawl9e/guys_i_finally_did_it/",
    "https://www.reddit.com/r/jobs/comments/1mt2apf/the_mental_health_cost_of_being_jobless_in_2025/",
    "https://www.reddit.com/r/jobs/comments/1mvpllc/im_going_to_start_sending_invoices_to_these/",
    "https://www.reddit.com/r/jobs/comments/1r2zhib/after_about_two_months_of_job_search_for_a_fresh/",
    "https://www.reddit.com/r/jobs/comments/1l2qaz0/after_4_months_and_150_applications_i_finally_got/",
    "https://www.reddit.com/r/jobs/comments/1nqsaul/finally_landed_a_job_after_10_months_and_1000/",
    "https://www.reddit.com/r/jobs/comments/1rks8hd/my_5week_job_search_after_being_laid_off_in_a/",
    "https://www.reddit.com/r/jobs/comments/1k7r8ez/gave_my_two_weeks_notice_then_got_ghosted/",
    "https://www.reddit.com/r/jobs/comments/1r3gvjy/has_anyone_else_lost_all_hope/",
    "https://www.reddit.com/r/jobs/comments/1mu0hsc/hiring_in_2025_the_new_hunger_games/"
]

In [9]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

rows = []

for url in jobs_links:
    json_url = url.rstrip("/") + ".json?raw_json=1"

    success = False

    for attempt in range(3):
        try:
            res = requests.get(json_url, headers=headers)

            print(res.status_code, url)

            if res.status_code == 200 and res.text.strip():
                data = res.json()
                success = True
                break
            else:
                time.sleep(3)

        except:
            time.sleep(3)

    if not success:
        print(f"Skipped: {url}")
        continue

    # still get post metadata for title/subreddit only
    try:
        post = data[0]["data"]["children"][0]["data"]
    except:
        continue

    # -------------------------
    # COMMENTS ONLY
    # -------------------------
    try:
        comments = data[1]["data"]["children"]
    except:
        comments = []

    for comment in comments:
        if comment.get("kind") != "t1":
            continue

        c = comment["data"]
        body = c.get("body", "").strip()

        if not body or body in ["[deleted]", "[removed]"]:
            continue

        rows.append({
            "Source Subreddit": post.get("subreddit", ""),
            "Post URL": url,
            "Post Title": post.get("title", ""),
            "Post or Comment": "comment",
            "Content (Full Text)": body,
            "Date Posted": datetime.fromtimestamp(c["created_utc"]).strftime("%m/%Y"),
            "Upvotes / Score": c.get("score", 0),
            "Theme Tag": "",
            "Platform Mentioned": "None",
            "Philippine Context?": "Yes" if post.get("subreddit", "").lower().startswith("ph") else "No",
            "Notable Quote": "",
            "Intern Notes": ""
        })

    time.sleep(4)

df = pd.DataFrame(rows)
df.to_csv("jobs_comments_only.csv", index=False, encoding="utf-8-sig")

print("Done. Rows collected:", len(df))

200 https://www.reddit.com/r/jobs/comments/1m6kdzc/no_one_is_hiring_me/
200 https://www.reddit.com/r/jobs/comments/1msay78/the_job_market_isnt_just_broken_its_breaking/
200 https://www.reddit.com/r/jobs/comments/1ly043a/i_got_fired_they_asked_me_to_give_my_100_at_the/
200 https://www.reddit.com/r/jobs/comments/1pddrfk/quit_job_and_was_ignored/
200 https://www.reddit.com/r/jobs/comments/1ldmqil/psa_dont_ignore_when_someone_offers_to_introduce/
200 https://www.reddit.com/r/jobs/comments/1m7c7nn/job_market_is_horrendous_us/
200 https://www.reddit.com/r/jobs/comments/1m5wl2z/getting_a_job_is_so_hard_right_now/
200 https://www.reddit.com/r/jobs/comments/1s4frb4/i_got_a_job_offer_and_im_devastated/
200 https://www.reddit.com/r/jobs/comments/1pgw7j9/how_should_i_respond_to_this_email_despite_my/
200 https://www.reddit.com/r/jobs/comments/1poxk1s/after_a_14_month_struggle_i_was_ready_to_give_it/
200 https://www.reddit.com/r/jobs/comments/1nis2km/my_wife_does_not_understand_the_state_of_the_job

## Comments for r/recruitinghell

In [10]:
recruitinghell_links = [
    "https://www.reddit.com/r/recruitinghell/comments/1mdk2a0/uniquely_awful_job_rejection_letter/",
    "https://www.reddit.com/r/recruitinghell/comments/1s5gc0r/apparently_bitching_to_hr_does_in_fact_work/",
    "https://www.reddit.com/r/recruitinghell/comments/1rmrboz/its_finally_over/",
    "https://www.reddit.com/r/recruitinghell/comments/1l7ghlb/ghosted_by_the_employer/",
    "https://www.reddit.com/r/recruitinghell/comments/1koo4u1/job_search_after_4000_applications/",
    "https://www.reddit.com/r/recruitinghell/comments/1n8lzjl/we_truly_are_in_hell/",
    "https://www.reddit.com/r/recruitinghell/comments/1prid7o/guys_i_did_it_by_lying/",
    "https://www.reddit.com/r/recruitinghell/comments/1mj5t8o/we_are_not_going_crazy/",
    "https://www.reddit.com/r/recruitinghell/comments/1l41t7n/10_totally_legit_reasons_why_getting_a_job_in/",
    "https://www.reddit.com/r/recruitinghell/comments/1pn5wrn/as_a_job_seeker_i_dont_think_theres_a_talent/",
    "https://www.reddit.com/r/recruitinghell/comments/1n4c5fm/executives_upset_that_they_cant_hold_on_to_top/",
    "https://www.reddit.com/r/recruitinghell/comments/1li0b0d/i_lost_2300_for_an_in_person_interview_that_was_a/",
    "https://www.reddit.com/r/recruitinghell/comments/1mk3m6o/this_really_confirms_what_we_suspected_all_along/",
    "https://www.reddit.com/r/recruitinghell/comments/1rmq7n1/ghosted_yesterday_8_minutes_late_today_i_hung_up/",
    "https://www.reddit.com/r/recruitinghell/comments/1rbmlxd/why_is_everyone_acting_like_were_not_in_the/",
    "https://www.reddit.com/r/recruitinghell/comments/1lmrlu4/gap_in_your_work_history_is_the_dumbest_thing_in/",
    "https://www.reddit.com/r/recruitinghell/comments/1oiiliq/some_amazon_recruiters_started_posting_on/",
    "https://www.reddit.com/r/recruitinghell/comments/1qdlg3v/i_wish_i_was_a_nepo_baby/",
    "https://www.reddit.com/r/recruitinghell/comments/1sckja7/after_90_interviews_i_got_an_offer/",
    "https://www.reddit.com/r/recruitinghell/comments/1rja5xc/software_engineering_is_now_a_closed_caste_if_you/",
    "https://www.reddit.com/r/recruitinghell/comments/1n736oq/hr_friend_told_me_half_of_these_vacancies_are/",
    "https://www.reddit.com/r/recruitinghell/comments/1n0kabu/i_fully_support_this_federal_legislation_to/",
    "https://www.reddit.com/r/recruitinghell/comments/1qwg2kf/billionaires_time_travel_to_preserve_the/",
    "https://www.reddit.com/r/recruitinghell/comments/1lpdyy7/i_have_lost_all_hope_in_humanity/",
    "https://www.reddit.com/r/recruitinghell/comments/1qajiu9/fmljust_got_this_tonight_i_hate_this_job_market/",
    "https://www.reddit.com/r/recruitinghell/comments/1pe74i8/if_i_can_get_sued_or_fired_for_lying_on_my_resume/",
    "https://www.reddit.com/r/recruitinghell/comments/1pp33z6/6_month_job_search_is_finally_over_fuck_this_job/",
    "https://www.reddit.com/r/recruitinghell/comments/1nl3pbk/finally_got_a_job_after_9_months_of_search/",
    "https://www.reddit.com/r/recruitinghell/comments/1kqlj7p/an_observation_after_six_months_of_unemployment/",
    "https://www.reddit.com/r/recruitinghell/comments/1rz6tta/when_your_job_hunt_is_so_bleak_your_dad_is_now/",
    "https://www.reddit.com/r/recruitinghell/comments/1ocqqhx/my_partner_was_rejected_from_a_job_bc_hes_been/",
    "https://www.reddit.com/r/recruitinghell/comments/1ofhpux/spent_my_friday_night_alone_crying_my_eyes_out/",
    "https://www.reddit.com/r/recruitinghell/comments/1r0k90o/insane_automated_email_i_got_after_i_applied_for/",
    "https://www.reddit.com/r/recruitinghell/comments/1kesvsc/i_have_become_the_enemy/",
    "https://www.reddit.com/r/recruitinghell/comments/1pgu203/all_of_these_are_for_minimum_wage_jobs/"
]

In [11]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

rows = []

for url in recruitinghell_links:
    json_url = url.rstrip("/") + ".json?raw_json=1"

    success = False

    for attempt in range(3):
        try:
            res = requests.get(json_url, headers=headers)

            print(res.status_code, url)

            if res.status_code == 200 and res.text.strip():
                data = res.json()
                success = True
                break
            else:
                time.sleep(3)

        except:
            time.sleep(3)

    if not success:
        print(f"Skipped: {url}")
        continue

    # still get post metadata for title/subreddit only
    try:
        post = data[0]["data"]["children"][0]["data"]
    except:
        continue

    # -------------------------
    # COMMENTS ONLY
    # -------------------------
    try:
        comments = data[1]["data"]["children"]
    except:
        comments = []

    for comment in comments:
        if comment.get("kind") != "t1":
            continue

        c = comment["data"]
        body = c.get("body", "").strip()

        if not body or body in ["[deleted]", "[removed]"]:
            continue

        rows.append({
            "Source Subreddit": post.get("subreddit", ""),
            "Post URL": url,
            "Post Title": post.get("title", ""),
            "Post or Comment": "comment",
            "Content (Full Text)": body,
            "Date Posted": datetime.fromtimestamp(c["created_utc"]).strftime("%m/%Y"),
            "Upvotes / Score": c.get("score", 0),
            "Theme Tag": "",
            "Platform Mentioned": "None",
            "Philippine Context?": "Yes" if post.get("subreddit", "").lower().startswith("ph") else "No",
            "Notable Quote": "",
            "Intern Notes": ""
        })

    time.sleep(4)

df = pd.DataFrame(rows)
df.to_csv("recruitinghell_comments_only.csv", index=False, encoding="utf-8-sig")

print("Done. Rows collected:", len(df))

200 https://www.reddit.com/r/recruitinghell/comments/1mdk2a0/uniquely_awful_job_rejection_letter/
200 https://www.reddit.com/r/recruitinghell/comments/1s5gc0r/apparently_bitching_to_hr_does_in_fact_work/
200 https://www.reddit.com/r/recruitinghell/comments/1rmrboz/its_finally_over/
503 https://www.reddit.com/r/recruitinghell/comments/1l7ghlb/ghosted_by_the_employer/
200 https://www.reddit.com/r/recruitinghell/comments/1l7ghlb/ghosted_by_the_employer/
200 https://www.reddit.com/r/recruitinghell/comments/1koo4u1/job_search_after_4000_applications/
200 https://www.reddit.com/r/recruitinghell/comments/1n8lzjl/we_truly_are_in_hell/
200 https://www.reddit.com/r/recruitinghell/comments/1prid7o/guys_i_did_it_by_lying/
200 https://www.reddit.com/r/recruitinghell/comments/1mj5t8o/we_are_not_going_crazy/
200 https://www.reddit.com/r/recruitinghell/comments/1l41t7n/10_totally_legit_reasons_why_getting_a_job_in/
200 https://www.reddit.com/r/recruitinghell/comments/1pn5wrn/as_a_job_seeker_i_dont_thi

## Comments for r/jobsearchhacks 

In [12]:
jobsearchhacks_links = [
    "https://www.reddit.com/r/jobsearchhacks/comments/1qxizav/i_stopped_getting_ghosted_by_recruiters_after_i/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1rnu44b/quit_my_job_with_no_offer_lined_up_because_of_my/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1m3mz41/finally_happened/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1kxlzc1/this_is_the_biggest_job_search_hack_ive_ever/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1ogfrhm/works_like_a_charm/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1me3u0m/it_seems_like_job_market_is_opening_up_a_bit/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1ktexgd/cold_messaging_on_linkedin_is_humbling_af/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1lyxtf7/my_partner_and_her_family_severely_underestimate/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1rkr4e3/45_months_250_applications_and_180_ghosts_later_i/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1o0fg6q/new_achievement_unlocked_leaving_this_subreddit/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1knzvj0/it_really_is_a_numbers_game/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1lv41nk/offer_after_10_months_of_unemployment_and_3/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1s8pk7w/i_started_treating_networking_events_like_coat/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1pbvevk/after_months_of_applying_here_is_what_finally/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1lmwu0e/how_to_find_a_job_in_this_current_job_market/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1n8o2mt/got_ghosted_by_a_startup_after_flying_out/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1shnmqa/some_job_search_hacks_that_are_actually_working/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1slvzst/i_went_from_no_jobs_opportunities_to_recruiters/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1pt75ak/i_got_hired/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1qit5fz/recruiters_kept_ghosting_me_until_i_started/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1mom796/after_6_months_i_finally_landed_a_job_hope_this/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1rx2kzm/i_sent_a_onepage_problem_breakdown_instead_of_a/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1lvgc1v/4_ways_to_maybe_get_a_job_before_it_even_shows_up/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1rw4n31/i_started_asking_what_usually_makes_people_get/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1s2eo3g/companies_are_posting_fake_jobs_to_figure_out_how/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1qn0s74/i_wish_linkedin_did_this/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1oblyv0/i_used_all_of_the_ai_job_application_autofill/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1pugxff/im_months_into_unemployment_and_i_feel_like_im/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1s4nmk5/why_does_applying_to_jobs_give_me_so_much_anxiety/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1sc304e/to_all_unemployed_job_seekers_how_are_you_doing/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1r27erc/why_am_i_not_being_hired/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1ryfloa/at_this_point_i_am_more_comfortable_with_the/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1k5bsmk/15_years_since_graduating_cannot_find_a_job_and/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1rzusjs/the_10second_rule_i_found_after_analyzing_100/",
    "https://www.reddit.com/r/jobsearchhacks/comments/1pt9xup/finally_a_great_offer/"
]

In [13]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

rows = []

for url in jobsearchhacks_links:
    json_url = url.rstrip("/") + ".json?raw_json=1"

    success = False

    for attempt in range(3):
        try:
            res = requests.get(json_url, headers=headers)

            print(res.status_code, url)

            if res.status_code == 200 and res.text.strip():
                data = res.json()
                success = True
                break
            else:
                time.sleep(3)

        except:
            time.sleep(3)

    if not success:
        print(f"Skipped: {url}")
        continue

    # still get post metadata for title/subreddit only
    try:
        post = data[0]["data"]["children"][0]["data"]
    except:
        continue

    # -------------------------
    # COMMENTS ONLY
    # -------------------------
    try:
        comments = data[1]["data"]["children"]
    except:
        comments = []

    for comment in comments:
        if comment.get("kind") != "t1":
            continue

        c = comment["data"]
        body = c.get("body", "").strip()

        if not body or body in ["[deleted]", "[removed]"]:
            continue

        rows.append({
            "Source Subreddit": post.get("subreddit", ""),
            "Post URL": url,
            "Post Title": post.get("title", ""),
            "Post or Comment": "comment",
            "Content (Full Text)": body,
            "Date Posted": datetime.fromtimestamp(c["created_utc"]).strftime("%m/%Y"),
            "Upvotes / Score": c.get("score", 0),
            "Theme Tag": "",
            "Platform Mentioned": "None",
            "Philippine Context?": "Yes" if post.get("subreddit", "").lower().startswith("ph") else "No",
            "Notable Quote": "",
            "Intern Notes": ""
        })

    time.sleep(4)

df = pd.DataFrame(rows)
df.to_csv("jobsearchhacks_comments_only.csv", index=False, encoding="utf-8-sig")

print("Done. Rows collected:", len(df))

200 https://www.reddit.com/r/jobsearchhacks/comments/1qxizav/i_stopped_getting_ghosted_by_recruiters_after_i/
200 https://www.reddit.com/r/jobsearchhacks/comments/1rnu44b/quit_my_job_with_no_offer_lined_up_because_of_my/
200 https://www.reddit.com/r/jobsearchhacks/comments/1m3mz41/finally_happened/
200 https://www.reddit.com/r/jobsearchhacks/comments/1kxlzc1/this_is_the_biggest_job_search_hack_ive_ever/
200 https://www.reddit.com/r/jobsearchhacks/comments/1ogfrhm/works_like_a_charm/
200 https://www.reddit.com/r/jobsearchhacks/comments/1me3u0m/it_seems_like_job_market_is_opening_up_a_bit/
200 https://www.reddit.com/r/jobsearchhacks/comments/1ktexgd/cold_messaging_on_linkedin_is_humbling_af/
200 https://www.reddit.com/r/jobsearchhacks/comments/1lyxtf7/my_partner_and_her_family_severely_underestimate/
200 https://www.reddit.com/r/jobsearchhacks/comments/1rkr4e3/45_months_250_applications_and_180_ghosts_later_i/
200 https://www.reddit.com/r/jobsearchhacks/comments/1o0fg6q/new_achievement_u